# Mobile Price Range Classification

**Tabular Classification Project**

## 2 · Project Overview

Classify mobile phones into 4 **price ranges** (0=low, 1=medium, 2=high, 3=very high) based on hardware specifications: RAM, battery, camera, screen, processor, etc. Synthetic dataset with 2,000 rows — a popular multiclass classification benchmark.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given 20 mobile phone specifications (RAM, battery, pixel resolution, weight, etc.), predict the price range (4 classes: 0-3).

## 5 · Why This Project Matters

- **Mobile phone pricing** is a multi-billion dollar market.
- This is a clean **4-class classification** with good separability.
- RAM is the dominant predictor — teaches feature importance analysis.
- 2,000 rows with 20 features is a good small-to-medium dataset.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | 2,000 |
| Features | 20 (battery_power, blue, clock_speed, dual_sim, fc, four_g, int_memory, m_dep, mobile_wt, n_cores, pc, px_height, px_width, ram, sc_h, sc_w, talk_time, three_g, touch_screen, wifi) |
| Target | `price_range` (4 classes: 0=low, 1=medium, 2=high, 3=very high) |
| Class balance | Equal (~25% each) |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Mobile Price Classification dataset (Kaggle).
**License**: Public / educational.
**Note**: Synthetic mobile phone specifications for classification practice.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "price_range"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Mobile Price Range Classification


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 2000

battery_power = np.random.randint(500, 2000, n)
blue = np.random.choice([0, 1], n)
clock_speed = np.random.uniform(0.5, 3.0, n).round(1)
dual_sim = np.random.choice([0, 1], n)
fc = np.random.randint(0, 20, n)
four_g = np.random.choice([0, 1], n)
int_memory = np.random.randint(2, 64, n)
m_dep = np.random.uniform(0.1, 1.0, n).round(1)
mobile_wt = np.random.randint(80, 200, n)
n_cores = np.random.randint(1, 9, n)
pc = np.random.randint(0, 20, n)
px_height = np.random.randint(0, 1960, n)
px_width = np.random.randint(500, 1998, n)
ram = np.random.randint(256, 3998, n)
sc_h = np.random.randint(5, 19, n)
sc_w = np.random.randint(0, 18, n)
talk_time = np.random.randint(2, 20, n)
three_g = np.random.choice([0, 1], n)
touch_screen = np.random.choice([0, 1], n)
wifi = np.random.choice([0, 1], n)

# Price range based primarily on RAM
score = ram + 0.3 * battery_power + 0.1 * px_height + np.random.normal(0, 200, n)
price_range = pd.qcut(score, q=4, labels=False)

df = pd.DataFrame({
    'battery_power': battery_power, 'blue': blue, 'clock_speed': clock_speed,
    'dual_sim': dual_sim, 'fc': fc, 'four_g': four_g, 'int_memory': int_memory,
    'm_dep': m_dep, 'mobile_wt': mobile_wt, 'n_cores': n_cores, 'pc': pc,
    'px_height': px_height, 'px_width': px_width, 'ram': ram, 'sc_h': sc_h,
    'sc_w': sc_w, 'talk_time': talk_time, 'three_g': three_g,
    'touch_screen': touch_screen, 'wifi': wifi, 'price_range': price_range,
})
print(f"Dataset shape: {df.shape}")
print(f"Price range distribution:\n{df['price_range'].value_counts().sort_index()}")
df.head()

Dataset shape: (2000, 21)
Price range distribution:
price_range
0    500
1    500
2    500
3    500
Name: count, dtype: int64


,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,1626,1,0.9,0,6,1,56,0.5,99,6,...,138,1881,3974,11,4,9,1,0,1,3
1,1959,0,0.9,1,16,1,52,0.1,145,3,...,102,1579,1419,6,15,13,0,1,1,1
2,1360,1,2.1,0,6,0,18,0.6,154,6,...,1265,1312,1944,17,2,12,0,1,1,1
3,1794,1,2.4,1,3,0,37,0.7,141,4,...,1140,1839,1819,8,10,7,0,0,1,1
4,1630,1,1.8,0,7,0,4,0.6,173,2,...,1237,1463,1706,9,14,14,1,1,0,1


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (2000, 21)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
price_range
3    500
1    500
0    500
2    500
Name: count, dtype: int64

Dtypes:
battery_power      int32
blue               int64
clock_speed      float64
dual_sim           int64
fc                 int32
four_g             int64
int_memory         int32
m_dep            float64
mobile_wt          int32
n_cores            int32
pc                 int32
px_height          int32
px_width           int32
ram                int32
sc_h               int32
sc_w               int32
talk_time          int32
three_g            int64
touch_screen       int64
wifi               int64
price_range        int64
dtype: object

Target 'price_range' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()

# Correlation with target
corr_target = df[num_cols].corrwith(df[TARGET]).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 8))
corr_target.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title("Feature Correlation with Price Range")
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(['ram', 'battery_power', 'px_height']):
    df.boxplot(column=col, by=TARGET, ax=axes[i])
    axes[i].set_title(f"{col} by Price Range")
plt.suptitle("")
plt.tight_layout()
plt.show()
print(f"Top 5 correlated features:\n{corr_target.head(5)}")

Top 5 correlated features:
ram              0.947641
battery_power    0.065459
n_cores          0.063214
four_g           0.041659
blue             0.041600
dtype: float64


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['steelblue', 'lightblue', 'coral', 'salmon'], edgecolor='black')
axes[0].set_title("Price Range Distribution")
axes[0].set_xticklabels(['Low (0)', 'Medium (1)', 'High (2)', 'Very High (3)'], rotation=0)
df[TARGET].value_counts().sort_index().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts().sort_index()}")

Class distribution:
price_range
0    500
1    500
2    500
3    500
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().sort_index().to_dict()}")

Train: (1600, 20), Test: (400, 20)
Train target dist: {0: 400, 1: 400, 2: 400, 3: 400}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
X_train = X_train.copy(); X_test = X_test.copy()
X_train['total_pixels'] = X_train['px_height'] * X_train['px_width']
X_test['total_pixels'] = X_test['px_height'] * X_test['px_width']
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean; X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (21): ['battery_power', 'blue', 'clock_speed', 'dual_sim', 'fc', 'four_g', 'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height', 'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'three_g', 'touch_screen', 'wifi', 'total_pixels']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
n_classes = len(np.unique(y_train))
if n_classes == 2:
    y_prob_bl = baseline.predict_proba(X_test)[:, 1]
else:
    y_prob_bl = None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.7400
Precision: 0.7403
Recall   : 0.7400
F1       : 0.7396


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                        
LinearDiscriminantAnalysis       0.8800             0.8800  0.981117  0.880197   0.880438  0.8800    0.032151
LogisticRegression               0.8700             0.8700  0.975967  0.870263   0.870648  0.8700    0.037835
GaussianNB                       0.8675             0.8675  0.975858  0.867861   0.870574  0.8675    0.024940
RandomForestClassifier           0.8675             0.8675  0.974787  0.867587   0.869191  0.8675    0.377106
CatBoostClassifier               0.8600             0.8600  0.976683  0.860861   0.863719  0.8600    4.231673
BaggingClassifier                0.8600             0.8600  0.966417  0.860281   0.861047  0.8600    0.096966
XGBClassifier                    0.8500             0.8500  0.972083  0.850319   0.851

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="macro_f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: catboost
Accuracy : 0.8850
F1       : 0.8851


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.8775  F1: 0.8779


LightGBM  Acc: 0.8450  F1: 0.8460


XGBoost   Acc: 0.8425  F1: 0.8425


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.8850  0.8851     0.8853  0.8850
CatBoost               0.8775  0.8779     0.8787  0.8775
LightGBM               0.8450  0.8460     0.8485  0.8450
XGBoost                0.8425  0.8425     0.8426  0.8425
Logistic Regression    0.7400  0.7396     0.7403  0.7400

Top 3: ['FLAML', 'CatBoost', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  FLAML
              precision    recall  f1-score   support

           0       0.93      0.91      0.92       100
           1       0.84      0.87      0.86       100
           2       0.86      0.84      0.85       100
           3       0.91      0.92      0.92       100

    accuracy                           0.89       400
   macro avg       0.89      0.89      0.89       400
weighted avg       0.89      0.89      0.89       400


  CatBoost
              precision    recall  f1-score   support

           0       0.94      0.90      0.92       100
           1       0.84      0.87      0.85       100
           2       0.83      0.84      0.84       100
           3       0.91      0.90      0.90       100

    accuracy                           0.88       400
   macro avg       0.88      0.88      0.88       400
weighted avg       0.88      0.88      0.88       400


  LightGBM
              precision    recall  f1-score   support



## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: FLAML
Error rate: 0.1150 (46 / 400)

Errors by true class:
      errors  total  error_rate
true                           
0          9    100        0.09
1         13    100        0.13
2         16    100        0.16
3          8    100        0.08


## 25 · Interpretation and Business Insight

- **RAM** is by far the strongest predictor of price range.
- **Battery power** and **pixel height** have moderate correlation.
- Binary features (bluetooth, wifi, 4G) have weak individual effects.
- Equal class distribution makes this a clean multiclass problem.
- Total screen pixels (height × width) is a useful engineered feature.

## 26 · Limitations

1. Synthetic data — real phone pricing depends on brand, marketing, supply chain.
2. No brand or model information.
3. Binary features (wifi, bluetooth) are nearly universal in real phones.
4. No time dimension — specifications evolve rapidly.
5. Price 'range' loses actual price information.

## 27 · How to Improve This Project

1. Add brand, release year, and market segment.
2. Use actual prices for regression instead of ranges.
3. Add customer review scores and sales volume.
4. Engineer feature interactions (screen quality = pixels × size).
5. Compare with ordinal regression (price ranges are ordered).

## 28 · Production Considerations

- Phone pricing models for market analysis and competitor benchmarking.
- Feature prioritization for phone manufacturers.
- Consumer recommendation based on budget.
- Regular retraining as market specifications evolve.

## 29 · Common Mistakes

1. Not recognizing RAM as the dominant predictor.
2. Treating 4 ordinal classes as completely unordered.
3. Not trying ordinal regression or ordered classifiers.
4. Including noise features that don't help (mobile weight, depth).
5. Over-engineering on a problem solved by 1-2 features.

## 30 · Mini Challenge / Exercises

1. Build a model with only RAM — what accuracy do you get?
2. Remove RAM and train — which feature steps up?
3. Try ordinal regression — does respecting class order help?
4. Identify the 5 most useless features and remove them.

## 31 · Final Summary / Key Takeaways

1. Mobile price classification is a well-separated 4-class problem.
2. RAM dominates price prediction — it alone gets ~85%+ accuracy.
3. Equal class distribution simplifies evaluation.
4. Ordinal nature of price ranges suggests ordered classifiers.
5. Real pricing depends on brand and market factors not captured here.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.8775,
    "f1": 0.8779,
    "precision": 0.8787,
    "recall": 0.8775
  },
  "LightGBM": {
    "accuracy": 0.845,
    "f1": 0.846,
    "precision": 0.8485,
    "recall": 0.845
  },
  "XGBoost": {
    "accuracy": 0.8425,
    "f1": 0.8425,
    "precision": 0.8426,
    "recall": 0.8425
  },
  "Logistic Regression": {
    "accuracy": 0.74,
    "f1": 0.7396,
    "precision": 0.7403,
    "recall": 0.74
  },
  "FLAML": {
    "accuracy": 0.885,
    "f1": 0.8851,
    "precision": 0.8853,
    "recall": 0.885
  }
}
